# 03. 화학물질 특성 예측 모델 구축

- 앞선 실습에서 HIF-2a 데이터를 모으고(01), 정리·분자 특성값을 계산함(02)
- 이번 시간에는 그 데이터로 **실제 예측 모델을 만들어 봄**
  - **분류 모델**: 화합물이 **활성(Active) / 비활성(Inactive)** 인지 맞힘
  - **회귀 모델**: **활성 점수(Score)를 숫자로** 예측함


---
## 0. 환경설정

- 실습에 필요한 도구(라이브러리)를 준비함
- Colab 환경에서는 대부분 이미 설치되어 있어 설치는 금방 끝남


In [ ]:
# [1] 필요한 라이브러리 설치 (Colab엔 대부분 이미 설치돼 있어서 금방 끝납니다)
!pip install scikit-learn matplotlib pandas

# [2] 데이터를 다루는 도구
import pandas as pd            # 표(엑셀 같은 데이터)를 다루는 도구
import numpy as np             # 숫자 계산을 도와주는 도구

# [3] 그림(그래프)을 그리는 도구
import matplotlib.pyplot as plt

# [4] 머신러닝(예측 모델) 도구들 - 필요한 것만 골라서 불러옵니다
from sklearn.model_selection import train_test_split          # 데이터를 학습/평가로 나누기
from sklearn.ensemble import RandomForestClassifier           # 분류 모델 (랜덤 포레스트)
from sklearn.ensemble import RandomForestRegressor            # 회귀 모델 (랜덤 포레스트)
from sklearn.metrics import accuracy_score, classification_report  # 분류 성능 지표
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # 혼동 행렬
from sklearn.metrics import roc_curve, auc                    # ROC 곡선
from sklearn.metrics import precision_recall_curve, average_precision_score  # PR 곡선
from sklearn.metrics import r2_score, mean_squared_error      # 회귀 성능 지표

print('준비 완료! 모든 도구를 불러왔습니다.')

## 1. 데이터 불러오기 & 특성/라벨 준비

- 지난 시간(02)에 만든 `hif2a_preprocessed.csv` 를 불러옴
- 이 파일에는 각 화합물의 **분자 특성값 6개**와 **활성 정보**가 들어 있음

| 구분 | 기호 | 의미 |
|---|---|---|
| 특성 | X (features) | 모델에게 보여줄 '입력' → 분자 특성값 6개 |
| 라벨 | y (label) | 모델이 맞혀야 할 '정답' → 활성 여부 (Active=1, Inactive=0) |


In [ ]:
# [1] 지난 시간(02 전처리 실습)에서 저장한 CSV 파일을 불러옵니다
df = pd.read_csv('hif2a_preprocessed.csv')

# [2] 데이터가 몇 줄, 몇 열인지 확인 (행 개수, 열 개수)
print('데이터 크기:', df.shape)

# [3] 맨 위 5줄을 미리 살펴보기
df.head()

In [ ]:
# [1] 모델의 입력이 될 분자 특성값 6개의 이름을 정리해 둡니다
feature_cols = ['MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds']

# [2] 특성(X): 위에서 정한 6개 열만 골라 담습니다
X = df[feature_cols]

# [3] 라벨(y): 'Active'이면 1, 아니면(Inactive) 0 으로 바꿔 줍니다
#     (컴퓨터는 글자보다 숫자를 더 잘 다루기 때문이에요)
y = (df['PUBCHEM_ACTIVITY_OUTCOME'] == 'Active').astype(int)

# [4] Active(1)와 Inactive(0)가 각각 몇 개인지 세어 봅니다 (데이터 균형 확인)
print('활성/비활성 개수:')
print(y.value_counts())

## 2. 데이터 분할 (학습용 / 평가용)

- 모델을 만들 때 데이터를 두 덩어리로 나눔

| 구분 | 용도 | 비율 |
|---|---|---|
| 학습용(train) | 모델이 공부하는 데 사용 | 80% |
| 평가용(test) | 처음 보는 데이터로 실력 시험 | 20% |

- 이렇게 나누는 이유: 공부한 문제만 잘 푸는 게 아니라 **처음 보는 문제도 잘 푸는지** 확인하려는 것임

> 🖼️ **[그림 자리]** — 전체 데이터 → train(80%) / test(20%) 분할 도식


In [ ]:
# [1] 데이터를 학습용(80%)과 평가용(20%)으로 나눕니다
#     - test_size=0.2  : 20%를 평가용으로
#     - random_state=42: 결과가 매번 똑같이 나오도록 하는 '고정 씨앗값'
#     - stratify=y     : Active/Inactive 비율을 양쪽에 똑같이 맞춰서 나누기
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# [2] 나눈 결과 확인 (몇 개씩 나뉘었는지)
print('학습용 개수:', len(X_train))
print('평가용 개수:', len(X_test))

## 3. 분류 모델 학습하기

- **랜덤 포레스트(Random Forest)** 모델을 사용함

> 랜덤 포레스트: 여러 개의 **결정 트리(나무)** 를 만들고, 각 나무의 예측을 모아 **다수결로 최종 답**을 정하는 방법임 (여러 명이 투표하는 것과 비슷함)

> 🖼️ **[그림 자리]** — 여러 결정 트리 → 다수결 투표 → 최종 예측 개념도


In [ ]:
# [1] 분류 모델을 만듭니다
#     - n_estimators=300 : 나무(결정트리)를 300그루 만들어서 다수결
#     - random_state=42  : 결과 재현을 위한 고정값
model = RandomForestClassifier(n_estimators=300, random_state=42)

# [2] 학습용 데이터로 모델을 '공부'시킵니다 (fit = 학습)
model.fit(X_train, y_train)

print('모델 학습 완료!')

## 4. 예측 & 성능 평가

- 학습이 끝난 모델에게 **평가용 데이터**를 보여줌
- 얼마나 잘 맞히는지 점수를 매김


In [ ]:
# [1] 평가용 데이터에 대해 예측 (Active/Inactive를 0 또는 1로)
y_pred = model.predict(X_test)

# [2] '활성일 확률'도 구합니다 ([:,1]은 Active(1)일 확률만 꺼내는 것)
#     -> 나중에 ROC, PR 곡선을 그릴 때 사용합니다
y_proba = model.predict_proba(X_test)[:, 1]

# [3] 정확도: 전체 중 몇 %를 맞혔는지
print('정확도(Accuracy): {:.3f}'.format(accuracy_score(y_test, y_pred)))

# [4] 더 자세한 성능표 (정밀도/재현율/F1 점수 등)
print('\n=== 상세 성능 보고서 ===')
print(classification_report(y_test, y_pred, target_names=['Inactive', 'Active']))

## 5. 성능 시각화

- 숫자만 보면 감이 잘 안 오므로, 성능을 **그림**으로 확인함

| 그림 | 보는 것 |
|---|---|
| Confusion Matrix (혼동 행렬) | 맞힌 것과 틀린 것을 표로 정리 |
| ROC 곡선 | 활성/비활성을 얼마나 잘 구분하는지 |
| PR 곡선 | 활성(양성)을 찾아내는 능력 |

> 참고: 그래프 축 글자 깨짐(한글 폰트 문제)을 막기 위해 **축 라벨은 영어**로 적음. 제목은 한글로 둬도 됨.


In [ ]:
# === Confusion Matrix (혼동 행렬) ===
# [1] 실제 정답(y_test)과 모델 예측(y_pred)을 비교한 표를 계산
cm = confusion_matrix(y_test, y_pred)

# [2] 한글 폰트 깨짐 방지를 위해 축 라벨은 영어로 표시합니다
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Inactive', 'Active'])

# [3] 그림 그리기 (숫자가 칸 안에 표시됩니다)
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Confusion Matrix')   # 제목
ax.set_xlabel('Predicted label')   # 가로축: 모델이 예측한 값
ax.set_ylabel('True label')        # 세로축: 실제 정답
plt.tight_layout()
plt.show()

In [ ]:
# === ROC 곡선 ===
# [1] 여러 기준점에서의 FPR(가짜 양성 비율)과 TPR(진짜 양성 비율)을 계산
fpr, tpr, _ = roc_curve(y_test, y_proba)

# [2] 곡선 아래 넓이(AUC) 계산 -> 1에 가까울수록 좋은 모델
roc_auc = auc(fpr, tpr)

# [3] 그림 그리기 (축 라벨은 폰트 깨짐 방지를 위해 영어로)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, color='C0', lw=2, label='ROC (AUC = {:.3f})'.format(roc_auc))
# 아무렇게나 찍는 모델의 기준선(대각선, AUC=0.50)
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random (0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# === PR 곡선 (Precision-Recall) ===
# [1] 여러 기준점에서의 정밀도(Precision)와 재현율(Recall)을 계산
precision, recall, _ = precision_recall_curve(y_test, y_proba)

# [2] 곡선의 평균 정밀도(AP) 계산 -> 1에 가까울수록 좋음
ap = average_precision_score(y_test, y_proba)

# [3] 기준선 = 실제 양성(Active) 비율 (아무 정보 없이 예측했을 때의 수준)
baseline = y_test.mean()

# [4] 그림 그리기 (축 라벨은 폰트 깨짐 방지를 위해 영어로)
plt.figure(figsize=(5, 5))
plt.plot(recall, precision, color='C1', lw=2, label='PR (AP = {:.3f})'.format(ap))
plt.axhline(y=baseline, color='gray', lw=1, linestyle='--',
            label='Baseline ({:.3f})'.format(baseline))
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 6. [실습] 회귀 모델로 활성 점수 예측하기

> **직접 해보기 🖐️**
>
> - 지금까지는 Active/Inactive를 맞히는 **분류**를 함
> - 이번에는 활성 점수(`Score`)라는 **숫자 자체를 예측**하는 **회귀** 모델을 직접 완성함
> - 아래 코드의 `______`(빈칸)을 채워 넣음 (그래프 등 나머지는 완성돼 있음)

💡 **힌트 — 분류 vs 회귀 대응**

| 항목 | 분류 | 회귀 |
|---|---|---|
| 모델 | `RandomForestClassifier` | `RandomForestRegressor` |
| 흐름 | `.fit()` → `.predict()` | 동일 |
| 지표 | `accuracy_score` | `r2_score` (1에 가까울수록 좋음) |


In [ ]:
# [1] 회귀의 정답(라벨)은 숫자 점수인 'Score' 입니다
y_reg = df['Score']

# [2] 특성(X)은 분류 때와 똑같이 6개 특성을 사용합니다
#     회귀용으로 데이터를 다시 학습/평가로 나눕니다 (숫자 예측이라 stratify는 사용 안 함)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
print('회귀 학습용:', len(Xr_train), '/ 회귀 평가용:', len(Xr_test))

In [ ]:
# ===== 여기 빈칸(______)을 채워 보세요! =====

# [1] 회귀 모델 만들기 (힌트: RandomForestRegressor, n_estimators=300, random_state=42)
reg = ______

# [2] 학습용 데이터로 모델 학습시키기 (힌트: 분류와 똑같이 .fit)
reg.______(Xr_train, yr_train)

# [3] 평가용 데이터로 점수 예측하기 (힌트: .predict)
yr_pred = reg.______(Xr_test)

# [4] 성능 평가하기
#     R2 점수 (1에 가까울수록 좋음)  -- 힌트: r2_score(실제값, 예측값)
r2 = ______(yr_test, yr_pred)
#     RMSE (0에 가까울수록 좋음) -- 오차의 크기, 이미 완성해 두었습니다
rmse = mean_squared_error(yr_test, yr_pred) ** 0.5

print('R2 점수: {:.3f}'.format(r2))
print('RMSE   : {:.3f}'.format(rmse))

In [ ]:
# === Parity plot: 실제값 vs 예측값 (이 코드는 이미 완성되어 있습니다) ===
# 점들이 대각선(y=x)에 가까울수록 예측이 정확한 것입니다
plt.figure(figsize=(5, 5))
plt.scatter(yr_test, yr_pred, alpha=0.5, color='C2')   # 실제(가로) vs 예측(세로)

# 완벽하게 맞혔을 때의 기준선 (y = x)
lo = min(yr_test.min(), yr_pred.min())
hi = max(yr_test.max(), yr_pred.max())
plt.plot([lo, hi], [lo, hi], color='gray', lw=1, linestyle='--', label='y = x')

plt.xlabel('Actual Score')       # 축 라벨은 폰트 깨짐 방지를 위해 영어
plt.ylabel('Predicted Score')
plt.title('Parity Plot (R2 = {:.3f})'.format(r2))
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# === Residual plot: 잔차(오차) 확인 (선택 사항, 이미 완성) ===
# 잔차 = 예측값 - 실제값. 0을 나타내는 가로선 근처에 고르게 흩어져 있으면 좋습니다
residuals = yr_pred - yr_test

plt.figure(figsize=(5, 4))
plt.scatter(yr_pred, residuals, alpha=0.5, color='C3')
plt.axhline(y=0, color='gray', lw=1, linestyle='--')
plt.xlabel('Predicted Score')
plt.ylabel('Residual (Pred - Actual)')
plt.title('Residual Plot')
plt.tight_layout()
plt.show()